# Blend with RealMLP — Cross-Family Ensemble

The key insight: **a 0.910 neural net can lift a 0.914 GBDT blend to 0.917+** if their errors are uncorrelated.

Strategy:
1. Load best GBDT submissions (XGBoost, LightGBM) + RealMLP
2. Measure prediction correlation — low correlation = high blend value
3. Grid search blend weights (GBDT heavy, RealMLP as diversity booster)
4. Generate prob + rank averaged submissions for each config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata
from itertools import product
import os

sns.set_theme(style='whitegrid')

IS_KAGGLE = os.path.exists('/kaggle/input')
SUB_DIR   = '/kaggle/working/' if IS_KAGGLE else '../'
DATA_DIR  = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'
TARGET    = 'Churn'

print(f'Submission dir: {SUB_DIR}')

## 1. Load Submissions

Three model families for maximum diversity:
- **XGBoost** — best single models (scaledpos, FE variants)
- **LightGBM** — different GBDT algorithm
- **RealMLP** — neural network (entirely different inductive bias)

In [ ]:
# ── Register all submissions with known LB scores ─────────────────────────────
SUBMISSIONS = [
    # XGBoost family
    {'file': 'submission_xgb_scaledpos_blend.csv',        'label': 'XGB-sp-blend',   'public_score': 0.91445},
    {'file': 'submission_xgb_scaledpos_ensemble.csv',     'label': 'XGB-sp-ens',     'public_score': 0.91444},
    {'file': 'submission_xgboost_tuned_multiseed_fe.csv', 'label': 'XGB-50t-FE',     'public_score': 0.91417},
    {'file': 'submission_xgboost_multiseed_fe_100_6.csv', 'label': 'XGB-100t-6fold', 'public_score': 0.91415},
    # LightGBM family
    {'file': 'submission_lgbm_tuned_multiseed_fe.csv',    'label': 'LGB-tuned-FE',   'public_score': None},
    {'file': 'submission_lightgbm_100n.csv',              'label': 'LGB-100n',       'public_score': None},
    # Neural network family
    {'file': 'submission_realmlp_tuned_multiseed_fe.csv', 'label': 'RealMLP',        'public_score': 0.91063},
    # EDA-driven FE
    {'file': 'submission_xgb_eda_fe.csv',                 'label': 'XGB-EDA-FE',     'public_score': None},
]

subs = {}
for s in SUBMISSIONS:
    path = os.path.join(SUB_DIR, s['file'])
    if os.path.exists(path):
        df = pd.read_csv(path)
        subs[s['label']] = {
            'preds': df[TARGET].values,
            'ids': df['id'].values,
            'score': s['public_score'],
            'file': s['file'],
        }
        score_str = f"{s['public_score']:.5f}" if s['public_score'] else 'unknown'
        print(f"  Loaded : {s['label']:20s} | LB: {score_str}")
    else:
        print(f"  MISSING: {s['file']}  ← place this file in {SUB_DIR}")

ref_ids = next(iter(subs.values()))['ids']
n = len(ref_ids)
print(f'\n{len(subs)} submissions loaded, {n:,} rows each.')

if 'RealMLP' not in subs:
    print('\n*** WARNING: RealMLP submission not found! ***')
    print('Copy submission_realmlp_tuned_multiseed_fe.csv to project root and re-run.')

## 2. Correlation Analysis

Lower correlation between models = higher blend value. RealMLP should show the lowest correlation with GBDTs.

In [ ]:
pred_df = pd.DataFrame({label: data['preds'] for label, data in subs.items()})
corr = pred_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlation heatmap
sns.heatmap(corr, annot=True, fmt='.4f', cmap='RdYlGn_r', vmin=0.85, vmax=1.0,
            ax=axes[0], square=True)
axes[0].set_title('Prediction Correlation (lower = more diverse)')

# Distribution comparison
for label in subs:
    axes[1].hist(subs[label]['preds'], bins=50, alpha=0.4, label=label, density=True)
axes[1].set_title('Prediction Distributions')
axes[1].set_xlabel('Predicted Probability')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Show RealMLP correlation with each GBDT model
if 'RealMLP' in subs:
    print('\nRealMLP correlation with GBDTs:')
    for label in subs:
        if label != 'RealMLP':
            r = corr.loc['RealMLP', label]
            print(f'  {label:20s}: {r:.4f}  {"← LOW (good diversity!)" if r < 0.97 else ""}')

## 3. Blend Utilities

In [ ]:
def weighted_blend(subs_dict, weights):
    """Weighted average of predictions (probability space)."""
    active = {k: v for k, v in weights.items() if v > 0 and k in subs_dict}
    total_w = sum(active.values())
    blended = np.zeros(n)
    for label, w in active.items():
        blended += (w / total_w) * subs_dict[label]['preds']
    return blended

def rank_blend(subs_dict, weights):
    """Rank-transform then weighted average."""
    active = {k: v for k, v in weights.items() if v > 0 and k in subs_dict}
    total_w = sum(active.values())
    blended = np.zeros(n)
    for label, w in active.items():
        ranks = rankdata(subs_dict[label]['preds']) / n
        blended += (w / total_w) * ranks
    return blended

def save_submission(preds, filename):
    """Save predictions as submission CSV."""
    sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
    sample_sub[TARGET] = preds
    path = os.path.join(SUB_DIR, filename)
    sample_sub.to_csv(path, index=False)
    return path

print('Blend utilities defined.')

## 4. Blend Configurations

Key principle: RealMLP gets a **small weight** (0.05-0.20) because its standalone score is lower.
The value comes from **diversity**, not raw performance. Even 5-10% weight from an uncorrelated model can help.

In [ ]:
# ── Blend Configs ─────────────────────────────────────────────────────────────
#
# Naming: A/B/C/D = increasing RealMLP weight
# All configs keep XGB-sp-blend as anchor (best single model at 0.91445)

BLEND_CONFIGS = {

    # A — Conservative: RealMLP at 5% — minimal risk
    'A_mlp_5pct': {
        'XGB-sp-blend':    3.0,
        'XGB-50t-FE':      2.0,
        'XGB-100t-6fold':  2.0,
        'LGB-100n':        1.5,
        'RealMLP':         0.5,   # ~5%
    },

    # B — Moderate: RealMLP at 10%
    'B_mlp_10pct': {
        'XGB-sp-blend':    3.0,
        'XGB-50t-FE':      2.0,
        'XGB-100t-6fold':  2.0,
        'LGB-100n':        1.5,
        'RealMLP':         1.0,   # ~10%
    },

    # C — Aggressive: RealMLP at 15%
    'C_mlp_15pct': {
        'XGB-sp-blend':    3.0,
        'XGB-50t-FE':      1.5,
        'XGB-100t-6fold':  1.5,
        'LGB-100n':        1.5,
        'RealMLP':         1.5,   # ~15%
    },

    # D — Max diversity: RealMLP at 20%
    'D_mlp_20pct': {
        'XGB-sp-blend':    3.0,
        'XGB-50t-FE':      1.5,
        'XGB-100t-6fold':  1.0,
        'LGB-100n':        1.5,
        'RealMLP':         2.0,   # ~20%
    },

    # E — Three-family equal: XGB / LGB / MLP each ~33%
    'E_three_equal': {
        'XGB-sp-blend':    1.5,
        'XGB-50t-FE':      1.5,
        'LGB-100n':        3.0,
        'RealMLP':         3.0,
    },

    # F — Best GBDT only (no RealMLP — control/baseline)
    'F_gbdt_only': {
        'XGB-sp-blend':    3.0,
        'XGB-50t-FE':      2.0,
        'XGB-100t-6fold':  2.0,
        'LGB-100n':        1.5,
    },

    # G — With LGB-tuned-FE if available
    'G_full_lineup': {
        'XGB-sp-blend':    3.0,
        'XGB-sp-ens':      1.0,
        'XGB-50t-FE':      1.5,
        'XGB-100t-6fold':  1.5,
        'LGB-tuned-FE':    2.0,
        'LGB-100n':        1.0,
        'RealMLP':         1.0,
    },
}

print('Blend configs:')
for name, weights in BLEND_CONFIGS.items():
    active = {k: v for k, v in weights.items() if v > 0}
    total = sum(active.values())
    mlp_pct = active.get('RealMLP', 0) / total * 100 if total > 0 else 0
    print(f"  {name:25s} | MLP: {mlp_pct:4.1f}% | models: {list(active.keys())}")

## 5. Generate All Blends

In [ ]:
results = []

print(f"{'Config':25s} | {'Type':5s} | {'Range':30s} | File")
print('-' * 100)

for config_name, weights in BLEND_CONFIGS.items():
    # Skip configs with missing models
    active = {k: v for k, v in weights.items() if v > 0}
    missing = [k for k in active if k not in subs]
    if missing:
        print(f"  {config_name:25s} | SKIP — missing: {missing}")
        continue

    for blend_type, blend_fn in [('prob', weighted_blend), ('rank', rank_blend)]:
        preds = blend_fn(subs, weights)
        fname = f"submission_realmlp_blend_{config_name}_{blend_type}.csv"
        path = save_submission(preds, fname)
        rng = f"[{preds.min():.4f}, {preds.max():.4f}]"
        print(f"  {config_name:25s} | {blend_type:5s} | {rng:30s} | {fname}")
        results.append({'config': config_name, 'type': blend_type, 'file': fname})

print(f"\n{len(results)} submission files generated.")

## 6. Weight Grid Search

Systematically search over RealMLP weight (0% to 30%) to find the sweet spot.
Since we don't have OOF predictions aligned across models, we compare prediction diversity metrics.

In [ ]:
if 'RealMLP' in subs:
    # Base GBDT weights (fixed)
    base_weights = {
        'XGB-sp-blend':    3.0,
        'XGB-50t-FE':      2.0,
        'XGB-100t-6fold':  2.0,
        'LGB-100n':        1.5,
    }
    base_total = sum(base_weights.values())

    # Sweep RealMLP weight from 0 to 3.0
    mlp_weights = np.arange(0.0, 3.1, 0.25)
    sweep_results = []

    # Reference: GBDT-only blend
    gbdt_only = weighted_blend(subs, base_weights)

    for mlp_w in mlp_weights:
        w = {**base_weights, 'RealMLP': mlp_w}
        total = base_total + mlp_w
        preds = weighted_blend(subs, w)

        # How much does the blend change from GBDT-only?
        diff_std = np.std(preds - gbdt_only)
        corr_with_gbdt = np.corrcoef(preds, gbdt_only)[0, 1]
        mlp_pct = mlp_w / total * 100 if total > 0 else 0

        sweep_results.append({
            'mlp_weight': mlp_w,
            'mlp_pct': mlp_pct,
            'diff_std': diff_std,
            'corr_with_gbdt_only': corr_with_gbdt,
            'pred_mean': preds.mean(),
            'pred_std': preds.std(),
        })

    sweep_df = pd.DataFrame(sweep_results)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(sweep_df['mlp_pct'], sweep_df['diff_std'], 'b-o', markersize=4)
    axes[0].set_xlabel('RealMLP Weight %')
    axes[0].set_ylabel('Std of Difference from GBDT-only')
    axes[0].set_title('Blend Diversity vs RealMLP Weight')
    axes[0].axvspan(5, 15, alpha=0.2, color='green', label='Sweet spot (5-15%)')
    axes[0].legend()

    axes[1].plot(sweep_df['mlp_pct'], sweep_df['corr_with_gbdt_only'], 'r-o', markersize=4)
    axes[1].set_xlabel('RealMLP Weight %')
    axes[1].set_ylabel('Correlation with GBDT-only Blend')
    axes[1].set_title('Blend Drift from GBDT-only')
    axes[1].axvspan(5, 15, alpha=0.2, color='green', label='Sweet spot (5-15%)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print('\nWeight sweep summary:')
    print(sweep_df.to_string(index=False, float_format='%.5f'))
else:
    print('RealMLP not loaded — skipping weight sweep.')

## 7. Recommended Submissions

Submit these in priority order (max 5 submissions/day on Kaggle):

In [ ]:
# Priority submissions to try
priority = [
    ('B_mlp_10pct', 'rank', 'Best bet — 10% MLP, rank averaged'),
    ('A_mlp_5pct',  'rank', 'Conservative — 5% MLP, rank averaged'),
    ('C_mlp_15pct', 'rank', 'Aggressive — 15% MLP, rank averaged'),
    ('B_mlp_10pct', 'prob', 'Prob blend for comparison'),
    ('F_gbdt_only', 'rank', 'Control — GBDT only (should match ~0.91433)'),
]

print('Recommended submission order:')
print('=' * 80)
for i, (config, blend_type, reason) in enumerate(priority, 1):
    fname = f'submission_realmlp_blend_{config}_{blend_type}.csv'
    exists = os.path.exists(os.path.join(SUB_DIR, fname))
    status = 'READY' if exists else 'MISSING'
    print(f'  {i}. [{status}] {fname}')
    print(f'     {reason}')
    if exists:
        print(f'     kaggle competitions submit -c playground-series-s6e3 -f {fname} -m "{reason}"')
    print()

## Summary

| Config | RealMLP % | Expected Outcome |
|--------|-----------|------------------|
| F (control) | 0% | ~0.91433 (reproduce current best) |
| A | 5% | Low risk, small diversity boost |
| **B** | **10%** | **Best bet — enough diversity without diluting GBDTs** |
| C | 15% | More aggressive, may overshoot |
| D | 20% | High risk — MLP score is 0.003+ lower |
| E | ~33% | Too much MLP — likely hurts |

**Key**: The sweet spot for blending a weaker-but-diverse model is typically **5-15%** weight.
If correlation between RealMLP and GBDTs is < 0.95, even 5% weight can give +0.001-0.003.